# Data Modeling — kiểm chứng thiết kế star schemaNotebook này **chứng minh** từng quyết định trong `star_schema.md`. Mỗi mục là một bài kiểm trachạy lại được trên dữ liệu nguồn.Thứ tự: grain/PK → cardinality → toàn vẹn FK → phi chuẩn hóa → thiết kế dimension → chất lượng dữ liệu.

In [ ]:
import pandas as pd, numpy as np
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 60)

DATA = 'data'
R = lambda f, **k: pd.read_csv(f'{DATA}/{f}', low_memory=False, **k)

cust   = R('customers.csv');  geo   = R('geography.csv');  prod = R('products.csv')
promo  = R('promotions.csv'); inv   = R('inventory.csv')
orders = R('orders.csv',   parse_dates=['order_date'])
items  = R('order_items.csv')
pay    = R('payments.csv')
ship   = R('shipments.csv', parse_dates=['ship_date','delivery_date'])
ret    = R('returns.csv',   parse_dates=['return_date'])
rev    = R('reviews.csv',   parse_dates=['review_date'])
wt     = R('web_traffic.csv', parse_dates=['date'])
sales  = R('sales.csv',     parse_dates=['Date'])
print('Đã nạp 13 bảng nguồn.')

---## 1 — Grain & khóa chính: cột nào thực sự unique?

In [ ]:
rows = []
for n, k, df in [('customers','customer_id',cust), ('geography','zip',geo), ('products','product_id',prod),
                 ('promotions','promo_id',promo), ('orders','order_id',orders), ('payments','order_id',pay),
                 ('shipments','order_id',ship), ('returns','return_id',ret), ('reviews','review_id',rev),
                 ('sales','Date',sales), ('web_traffic','date',wt)]:
    rows.append([n, k, len(df), df[k].nunique(), 'OK' if df[k].is_unique else '*** TRÙNG ***'])
for n, ks, df in [('order_items','order_id+product_id',items), ('inventory','snapshot_date+product_id',inv)]:
    d = df.duplicated(ks.split('+')).sum()
    rows.append([n, ks, len(df), len(df)-d, 'OK' if d==0 else f'*** {d} DÒNG TRÙNG ***'])
print(pd.DataFrame(rows, columns=['bảng','khóa ứng viên','số dòng','số khóa','kết luận']).to_string(index=False))

In [ ]:
dup = items[items.duplicated(['order_id','product_id'], keep=False)].sort_values(['order_id','product_id'])
print(f'{len(dup)} dòng thuộc {dup.order_id.nunique()} đơn có (order_id, product_id) trùng:\n')
print(dup.head(8).to_string(index=False))
print('\n→ Cùng sản phẩm, cùng đơn, nhưng SỐ LƯỢNG và GIÁ khác nhau — là hai dòng hàng riêng biệt.')
print('→ Khóa tổ hợp KHÔNG hợp lệ. fact_order_item BẮT BUỘC dùng surrogate key + line_number.')

---## 2 — Cardinality: quan hệ giữa các bảng

In [ ]:
rows = []
for nb, kb, b in [('order_items','order_id',items), ('payments','order_id',pay), ('shipments','order_id',ship),
                  ('returns','order_id',ret), ('reviews','order_id',rev)]:
    mx = b.groupby(kb).size().max()
    cov = b[kb].nunique()
    rows.append([f'orders → {nb}', f'{len(orders):,}', f'{cov:,}', mx,
                 '1:1' if mx==1 and cov==len(orders) else ('1:0..1' if mx==1 else f'1:0..{mx}')])
print(pd.DataFrame(rows, columns=['quan hệ','số đơn','số đơn được phủ','max dòng/đơn','cardinality']).to_string(index=False))
print('\n→ payments là 1:1 ĐẦY ĐỦ với orders  → ứng viên gộp vào fact_order (xem mục 4).')
print('→ returns/reviews là 1:0..N          → bảng fact riêng.')
print('→ shipments 1:0..1, chỉ phủ 566.067/646.945 → cột nullable trong accumulating snapshot.')

In [ ]:
# returns/reviews tham chiếu ĐƠN hay DÒNG HÀNG?
key = set(map(tuple, items[['order_id','product_id']].values))
for n, df in [('returns', ret), ('reviews', rev)]:
    miss = sum(1 for t in map(tuple, df[['order_id','product_id']].values) if t not in key)
    print(f'  {n}: {miss:,}/{len(df):,} dòng có (order_id, product_id) KHÔNG tồn tại trong order_items')
print('\n→ Khớp 100% ở mức (đơn, sản phẩm) → chúng tham chiếu DÒNG HÀNG, không phải đơn.')
print('→ FK phải trỏ tới fact_order_item.order_item_sk.')
print('→ NHƯNG: với 16 khóa tự nhiên trùng ở mục 1, join theo khóa tự nhiên là NHẬP NHẰNG')
print('   → ETL phải gán line_number ổn định trước khi join.')

---## 3 — Toàn vẹn khóa ngoại

In [ ]:
def fk(child, ck, parent, pk, nc, np_):
    orphan = (~child[ck].isin(parent[pk])).sum()
    unused = (~parent[pk].isin(child[ck])).sum()
    return [f'{nc}.{ck}', f'{np_}.{pk}', orphan, f'{orphan/len(child)*100:.2f}%', unused]

checks = [
    fk(orders,'customer_id',cust,'customer_id','orders','customers'),
    fk(orders,'zip',geo,'zip','orders','geography'),
    fk(cust,'zip',geo,'zip','customers','geography'),
    fk(items,'order_id',orders,'order_id','order_items','orders'),
    fk(items,'product_id',prod,'product_id','order_items','products'),
    fk(pay,'order_id',orders,'order_id','payments','orders'),
    fk(ship,'order_id',orders,'order_id','shipments','orders'),
    fk(ret,'order_id',orders,'order_id','returns','orders'),
    fk(ret,'product_id',prod,'product_id','returns','products'),
    fk(rev,'order_id',orders,'order_id','reviews','orders'),
    fk(rev,'customer_id',cust,'customer_id','reviews','customers'),
    fk(rev,'product_id',prod,'product_id','reviews','products'),
    fk(inv,'product_id',prod,'product_id','inventory','products'),
    fk(items[items.promo_id.notna()],'promo_id',promo,'promo_id','order_items','promotions'),
]
t = pd.DataFrame(checks, columns=['khóa ngoại','tham chiếu','orphan','tỷ lệ','cha không được dùng'])
print(t.to_string(index=False))
print(f'\n→ TỔNG ORPHAN = {t.orphan.sum()} trên 14 quan hệ. Toàn vẹn tham chiếu hoàn hảo.')
print('→ Nhưng nhiều bản ghi cha không được dùng: 31.684 KH chưa mua, 814 SP chưa bán.')

---## 4 — Phi chuẩn hóa: bản sao có nhất quán không?Nếu bản sao **nhất quán 100%** thì đó là dư thừa thuần túy → bỏ được an toàn.

In [ ]:
rows = []
j = orders[['order_id','payment_method']].merge(pay[['order_id','payment_method']], on='order_id', suffixes=('_o','_p'))
rows.append(['orders.payment_method', 'payments.payment_method', (j.payment_method_o!=j.payment_method_p).sum(), len(j)])

j = cust[['zip','city']].merge(geo[['zip','city']], on='zip', suffixes=('_c','_g'))
rows.append(['customers.city', 'geography.city', (j.city_c!=j.city_g).sum(), len(j)])

j = orders[['customer_id','zip']].merge(cust[['customer_id','zip']], on='customer_id', suffixes=('_o','_c'))
rows.append(['orders.zip', 'customers.zip', (j.zip_o!=j.zip_c).sum(), len(j)])

ji = inv[['product_id','product_name','category','segment']].drop_duplicates().merge(
     prod[['product_id','product_name','category','segment']], on='product_id', suffixes=('_i','_p'))
for c in ['product_name','category','segment']:
    rows.append([f'inventory.{c}', f'products.{c}', (ji[c+'_i']!=ji[c+'_p']).sum(), len(ji)])

j = rev[['order_id','customer_id']].merge(orders[['order_id','customer_id']], on='order_id', suffixes=('_r','_o'))
rows.append(['reviews.customer_id', 'orders.customer_id', (j.customer_id_r!=j.customer_id_o).sum(), len(j)])

t = pd.DataFrame(rows, columns=['cột sao chép','nguồn gốc','số dòng lệch','tổng dòng'])
print(t.to_string(index=False))
print(f'\n→ TỔNG LỆCH = {t["số dòng lệch"].sum()}. Mọi bản sao đều nhất quán tuyệt đối → bỏ được.')
print('\n→ ĐẶC BIỆT: orders.zip == customers.zip ở 100% dòng.')
print('   Đây KHÔNG phải địa chỉ giao hàng riêng → KHÔNG có ship-to role-playing dimension.')
print('   dim_geography chỉ tiếp cận qua dim_customer.')

In [ ]:
# payments có suy ra được hoàn toàn không?
gross = items.assign(g=items.quantity*items.unit_price).groupby('order_id').g.sum()
net   = items.assign(n=items.quantity*items.unit_price-items.discount_amount).groupby('order_id').n.sum()
p = pay.set_index('order_id')
print(f'payment_value == tổng GROSS : khớp {np.isclose(p.payment_value, p.join(gross).g).mean()*100:6.2f}%')
print(f'payment_value == tổng NET   : khớp {np.isclose(p.payment_value, p.join(net).n).mean()*100:6.2f}%')
print(f'\ninstallments: {sorted(pay.installments.unique())}')
print('\n→ payment_value SUY RA ĐƯỢC hoàn toàn từ order_items (net). payment_method thì trùng orders.')
print('→ Chỉ `installments` là thông tin mới → gộp vào fact_order, XÓA bảng payments.')
print('\n⚠ NGỮ NGHĨA: payments dùng NET, nhưng sales.csv tính Revenue theo GROSS.')
print('  Hai định nghĩa doanh thu cùng tồn tại — phải chọn 1 và ghi rõ trong metric layer.')

---## 5 — Thiết kế dimension

In [ ]:
print('5.1 — JUNK DIMENSION\n')
lc = ['order_status','payment_method','device_type','order_source']
for c in lc:
    print(f'  {c:16s} {orders[c].nunique()} giá trị')
cart = np.prod([orders[c].nunique() for c in lc])
actual = len(orders[lc].drop_duplicates())
print(f'\n  tích Descartes = {cart} | tổ hợp thực tế xuất hiện = {actual}')
print(f'  → gộp 4 cột thành 1 dim_order_junk ({actual} dòng) thay vì 4 dimension rời.')
print(f'  → {actual}/{cart} tổ hợp đều xuất hiện: dấu hiệu dữ liệu sinh tổng hợp.')

In [ ]:
print('5.2 — dim_geography: có phân cấp snowflake không?\n')
print(f'  zip={geo.zip.nunique():,} → city={geo.city.nunique()} | district={geo.district.nunique()} | region={geo.region.nunique()}')
print('\n  kiểm tra tính lồng nhau:')
for a, b in [('district','region'), ('city','region'), ('city','district'), ('district','city')]:
    n = geo.groupby(a)[b].nunique()
    ok = 'CÓ lồng' if n.max()==1 else f'KHÔNG (1 {a} → {n.min()}..{n.max()} {b})'
    print(f'    {a:9s} → {b:9s}: {ok}')
print('\n  → city và district ĐỀU roll-up về region, nhưng CẮT CHÉO nhau.')
print('  → Không tồn tại một cây snowflake duy nhất → giữ dim_geography PHẲNG ở grain = zip.')
print('  → Vì chỉ có 42/39/3 giá trị, có thể denormalize thẳng vào dim_customer (tiết kiệm 1 join).')

In [ ]:
print('5.3 — review_title có phải junk dimension không?\n')
ct = pd.crosstab(rev.rating, rev.review_title)
n_rating_per_title = (ct > 0).sum(axis=0)   # mỗi TITLE xuất hiện ở bao nhiêu rating
n_title_per_rating = (ct > 0).sum(axis=1)   # mỗi RATING dùng bao nhiêu title
print(f'  review_title: {rev.review_title.nunique()} giá trị / rating: {rev.rating.nunique()} giá trị')
print(f'  số title xuất hiện ở >1 rating : {(n_rating_per_title > 1).sum()}/{len(n_rating_per_title)}'
      f'   → FD  review_title → rating  ĐÚNG')
print(f'  số title mỗi rating dùng      : {n_title_per_rating.to_dict()}'
      f'   → FD  rating → review_title  SAI')
print('\n  → CHIỀU ĐÚNG là review_title → rating: mỗi title thuộc đúng MỘT rating,')
print('    nhưng một rating dùng 3–4 title khác nhau. KHÔNG phải chiều ngược lại.')
print('  → Vậy bảng tra cứu phải khóa theo TITLE và có 18 dòng —')
print('    KHÔNG phải dim_rating 5 dòng. Xem normalization.ipynb §4.3.')
print('  → Trong star schema: fact_review giữ `rating` (measure gộp được), bỏ `review_title`.')
print('    Đây là phi chuẩn hóa CÓ CHỦ ĐÍCH, không mâu thuẫn với FD ở trên.')
print('  → Đừng gộp chung với dim_order_junk.')

In [ ]:
print('5.4 — SCD: có dựng được Type 2 không?\n')
j = items.merge(prod[['product_id','price']], on='product_id')
print(f'  order_items.unit_price == products.price : khớp {np.isclose(j.unit_price, j.price).mean()*100:.2f}%')
pu = j.groupby('product_id').unit_price.nunique()
print(f'  số giá khác nhau mỗi sản phẩm: trung vị={pu.median():.0f}, max={pu.max():,}')
print('  → products.price chỉ là SNAPSHOT hiện tại, không phải giá lịch sử.')
print('  → Đáng lẽ cần SCD Type 2, NHƯNG nguồn KHÔNG có cột effective date nào.')
print('  → Quyết định: dim_product Type 1 + LƯU unit_price TRONG FACT.\n')

jo = orders.merge(cust[['customer_id','signup_date']], on='customer_id')
jo['signup_date'] = pd.to_datetime(jo.signup_date)
bad = jo.order_date < jo.signup_date
print(f'  đơn phát sinh TRƯỚC ngày signup: {bad.sum():,}/{len(jo):,} ({bad.mean()*100:.1f}%)')
print(f'  độ lệch trung vị: {(jo.signup_date-jo.order_date)[bad].dt.days.median():.0f} ngày')
print('  → signup_date sinh NGẪU NHIÊN, độc lập lịch sử đơn → không dùng tính tenure/cohort.')
print('  → dim_customer buộc phải Type 1 (không có lịch sử thuộc tính nào trong nguồn).')

In [ ]:
print('5.5 — quan hệ M:N với promotions\n')
print(f'  promo_id   có giá trị: {items.promo_id.notna().sum():,} ({items.promo_id.notna().mean()*100:.1f}%)')
print(f'  promo_id_2 có giá trị: {items.promo_id_2.notna().sum():,} ({items.promo_id_2.notna().mean()*100:.2f}%)')
print(f'  giá trị của promo_id_2: {list(items.promo_id_2.dropna().unique())}')
print('\n  → 2 cột lặp promo_id/promo_id_2 là quan hệ M:N bị nhét vào cột lặp (vi phạm 1NF).')
print('  → Chuẩn hóa thành bridge_item_promo, dù cột thứ 2 gần như không được dùng.')

# kiem chung khoa chinh cua bridge KHONG the dung do
p2 = items[items.promo_id_2.notna()]
print(f'\n  KIỂM CHỨNG PK (order_item_sk, promotion_sk):')
print(f'    dòng có promo_id == promo_id_2 : {(p2.promo_id==p2.promo_id_2).sum()}  (phải = 0)')
print(f'    dòng có promo_id NULL khi có promo_id_2: {p2.promo_id.isna().sum()}  (phải = 0)')
print(f'    → PK hợp lệ. Kích thước bridge = {items.promo_id.notna().sum():,} + {len(p2)} = '
      f'{items.promo_id.notna().sum()+len(p2):,} dòng')

print('\n  Các cặp promo đồng thời:')
print(p2.groupby(['promo_id','promo_id_2']).size().to_string())
pm = R('promotions.csv', parse_dates=['start_date','end_date']).set_index('promo_id')
for a, b in p2.groupby(['promo_id','promo_id_2']).size().index:
    lo = max(pm.loc[a,'start_date'], pm.loc[b,'start_date'])
    hi = min(pm.loc[a,'end_date'],   pm.loc[b,'end_date'])
    print(f'    {a} ({pm.loc[a,"promo_name"]}) + {b} ({pm.loc[b,"promo_name"]})')
    print(f'      → chồng lấn {lo.date()} .. {hi.date()}')
print('\n  → promo_id_2 CHỈ xuất hiện trong cửa sổ chồng lấn của 2 promo — không phải cột tùy tiện.')

---## 6 — `fact_inventory_snapshot` là periodic snapshot (bán cộng tính)

In [ ]:
inv['snapshot_date'] = pd.to_datetime(inv.snapshot_date)
print(f'  grain (snapshot_date, product_id) unique : {not inv.duplicated(["snapshot_date","product_id"]).any()}')
print(f'  số mốc: {inv.snapshot_date.nunique()} | tất cả là cuối tháng: {inv.snapshot_date.dt.is_month_end.all()}')
print(f'  độ phủ sản phẩm: {inv.product_id.nunique():,}/{prod.product_id.nunique():,}')

mo = items.merge(orders[['order_id','order_date']], on='order_id')
mo['ym'] = mo.order_date.dt.to_period('M')
real = mo.groupby(['ym','product_id']).quantity.sum().rename('thực_tế')
inv['ym'] = inv.snapshot_date.dt.to_period('M')
cmp_ = inv.set_index(['ym','product_id']).join(real)
print(f'\n  inventory.units_sold khớp số bán thực tế: {np.isclose(cmp_.units_sold, cmp_["thực_tế"].fillna(-1)).mean()*100:.2f}%')
print('\n  → PERIODIC SNAPSHOT FACT (mốc cuối tháng).')
print('  → BÁN CỘNG TÍNH: stock_on_hand và days_of_supply KHÔNG được SUM qua nhiều ngày,')
print('    chỉ cộng được theo sản phẩm trong CÙNG một mốc. Qua thời gian dùng AVG hoặc mốc cuối.')
print('  → units_sold KHÔNG reconcile với fact_order_item → nguồn ĐỘC LẬP, đừng đối soát.')
print('  → Bỏ product_name/category/segment (trùng products 100%), thay bằng FK product_sk.')

---## 7 — `fact_daily_sales` là aggregate factXem `eda.ipynb` mục 1 để có chứng minh đầy đủ. Ở đây chỉ xác nhận lại kết luận.

In [ ]:
m = items.merge(orders[['order_id','order_date']], on='order_id').merge(prod[['product_id','cogs']], on='product_id')
g = m.assign(gross=m.quantity*m.unit_price, ca=m.quantity*m.cogs).groupby('order_date').agg(
    rev=('gross','sum'), cg=('ca','sum'))
j = sales.set_index('Date').join(g)
print(f'  sai số tương đối lớn nhất — Revenue: {((j.Revenue-j.rev).abs()/j.Revenue).max():.2e}')
print(f'  sai số tương đối lớn nhất — COGS   : {((j.COGS-j.cg).abs()/j.COGS).max():.2e}')
print('\n  → fact_daily_sales là BẢNG TỔNG HỢP suy ra từ fact_order_item, không phải nguồn sự thật độc lập.')
print('  → Nhưng là fact DUY NHẤT phải mở rộng quá 2022-12-31 để chứa 548 dòng dự báo.')
print('  → Dùng cờ is_actual để phân biệt thực tế / dự báo.')

---## 8 — Vấn đề chất lượng dữ liệu → quy tắc kiểm tra khi load

In [ ]:
print('8.1 — products.price có HAI hệ đơn vị\n')
# Ranh giới thật là 100, KHÔNG phải 1000. Ngưỡng 1000 gộp nhầm hai quần thể khác hẳn nhau:
# 654 SKU chưa từng bán (lỗi dữ liệu) và 34 SKU bán bình thường (hàng rẻ hợp lệ).
sold  = set(items.product_id.unique())
slices = [('price < 100',        prod.price < 100),
          ('100 ≤ price < 1000', prod.price.between(100, 1000, inclusive='left')),
          ('price ≥ 1000',       prod.price >= 1000)]
prod['m'] = 1 - prod.cogs/prod.price
print(f'  {"lát cắt":20s}{"SKU":>6s}{"từng bán":>10s}{"giá min":>12s}{"giá max":>12s}{"biên LN TB":>12s}')
for name, mask in slices:
    s = prod[mask]
    print(f'  {name:20s}{mask.sum():>6d}{s.product_id.isin(sold).sum():>10d}'
          f'{s.price.min():>12.2f}{s.price.max():>12.2f}{s.m.mean():>12.3f}')

ji = items.merge(prod[['product_id','price']], on='product_id')
ji['g'] = ji.quantity*ji.unit_price
lo100, mid = ji.price < 100, ji.price.between(100, 1000, inclusive='left')
print(f'\n  dòng bán có price < 100        : {lo100.mean()*100:.2f}% số dòng, {ji[lo100].g.sum()/ji.g.sum()*100:.2f}% doanh thu')
print(f'  dòng bán có 100 ≤ price < 1000 : {mid.mean()*100:.2f}% số dòng, {ji[mid].g.sum()/ji.g.sum()*100:.2f}% doanh thu')
print(f'\n  → 654 SKU giá < 100 KHÔNG có lấy một dòng bán nào ⇒ đây mới là lỗi dữ liệu.')
print(f'    Toàn bộ 7,46% / 1,06% ở trên đến từ 30 SKU thuộc lát cắt GIỮA, và chúng bán ở')
print(f'    ĐÚNG giá niêm yết: unit_price/price trung vị {(ji[mid].unit_price/ji[mid].price).median():.3f}, '
      f'unit_price trung vị {ji[mid].unit_price.median():.2f}')
print('    ⇒ hàng rẻ HỢP LỆ, không phải giá sai. Ngưỡng cờ price_anomaly phải là 100, không phải 1000.')

d = prod[prod.duplicated(['product_name','size','color'], keep=False)].sort_values(['product_name','size','color'])
print(f'\n  {len(d)} dòng có (name,size,color) trùng — CÙNG sản phẩm ở CẢ HAI thang giá:')
print(d.head(4)[['product_id','product_name','size','color','price']].to_string(index=False))
print(f'    (trong đó {(d.price < 100).sum()} dòng rơi vào lát cắt < 100, và {d[d.price<100].product_id.isin(sold).sum()} dòng trong số đó từng bán)')

prod['blk'] = (prod.product_id//500)*500
print('\n  phân bố theo khối product_id (xen kẽ hay thành khối?):')
print(pd.crosstab(prod.blk, (prod.price < 100).rename('giá<100')).to_string())
print('\n  → XEN KẼ theo product_id, không thành khối → LỖI DỮ LIỆU, không phải trục phân tích.')
print('  → ĐỪNG tạo thuộc tính price_tier. Đặt rule cảnh báo khi load: price < 100.')

In [ ]:
print('8.2 — mâu thuẫn giữa order_status và bảng returns\n')
n_ret_status = (orders.order_status=='returned').sum()
print(f"  đơn có status 'returned'        : {n_ret_status:,}")
print(f"  đơn xuất hiện trong bảng returns: {ret.order_id.nunique():,}")
print(f'  chênh lệch: {n_ret_status - ret.order_id.nunique():,} đơn')
print('  → HAI nguồn sự thật mâu thuẫn. Phải chọn 1 và ghi rõ trong metric layer.\n')

print('8.3 — đơn thiếu bản ghi shipment\n')
noship = orders[~orders.order_id.isin(ship.order_id)]
vc = noship.order_status.value_counts()
print(vc.to_string())
print(f"\n  → cancelled/paid/created không gửi hàng là HỢP LÝ.")
print(f"  → NHƯNG {vc.get('delivered',0)} đơn 'delivered', {vc.get('shipped',0)} đơn 'shipped',")
print(f"     {vc.get('returned',0)} đơn 'returned' KHÔNG có shipment → lỗi tính đầy đủ.")

In [ ]:
print('8.4 — tổng hợp cột chết & cột phi thực tế\n')
for n, df in [('inventory',inv), ('order_items',items), ('customers',cust), ('orders',orders), ('promotions',promo)]:
    dead   = [c for c in df.columns if df[c].nunique(dropna=True) <= 1]
    empty  = [f'{c} ({df[c].isna().mean()*100:.2f}% null)' for c in df.columns if df[c].isna().mean() > 0.9]
    if dead or empty:
        print(f'  {n:12s} hằng số={dead} | gần như rỗng={empty}')
print(f'\n  web_traffic: {len(wt):,} dòng / {wt.date.nunique():,} ngày = 1 dòng/ngày,')
print(f'    nhưng traffic_source có {wt.traffic_source.nunique()} giá trị → KHÔNG phải phân rã theo nguồn,')
print(f'    mỗi ngày mang 1 nhãn xoay vòng. bounce_rate ≈ {wt.bounce_rate.mean():.4f} → phi thực tế.')

# return_quantity <= quantity: KẾT QUẢ PHỤ THUỘC KHÓA JOIN.
# Join bằng (order_id, product_id) là SAI — mục 1 đã chứng minh đó không phải khóa.
# Phải gán line_number theo thứ tự nguồn rồi join, đúng như mục 2 đã cảnh báo.
it = items.assign(line_number=items.groupby('order_id').cumcount() + 1)
it['_k'] = it.groupby(['order_id','product_id']).cumcount()
rt = ret.assign(_k=ret.groupby(['order_id','product_id']).cumcount())
z2 = rt.merge(it[['order_id','product_id','_k','line_number','quantity']],
              on=['order_id','product_id','_k'], how='left')
print(f'\n  return_quantity > quantity, join đúng khóa (order_id, line_number):')
print(f'    {(z2.return_quantity > z2.quantity).sum()} vi phạm / {len(z2):,} dòng (không fan-out)')
print('    → CẢNH BÁO GIẢ, KHÔNG chặn dòng nào khi load.')
print('    → Chứng minh đầy đủ cả hai nhánh join: normalization.ipynb §6 / normalized_schema.md §8.')

---## 9 — Quy mô các bảng sau khi mô hình hóa

In [ ]:
import datetime
n_date = (pd.Timestamp('2024-07-01') - pd.Timestamp('2012-01-01')).days + 1
sizes = [
    ('dim_date', n_date, 'phủ cả vùng dự báo tới 2024-07-01'),
    ('dim_customer', len(cust), 'SCD Type 1'),
    ('dim_product', len(prod), 'SCD Type 1, giá lưu trong fact'),
    ('dim_geography', len(geo), 'phẳng, grain = zip'),
    ('dim_promotion', len(promo), ''),
    ('dim_order_junk', len(orders[['order_status','payment_method','device_type','order_source']].drop_duplicates()), 'đủ tích Descartes'),
    ('fact_order_item', len(items), 'transaction, surrogate key'),
    ('fact_order', len(orders), 'accumulating snapshot'),
    ('fact_daily_sales', len(sales)+548, 'aggregate, gồm 548 dòng dự báo'),
    ('fact_inventory_snapshot', len(inv), 'periodic snapshot, BÁN CỘNG TÍNH'),
    ('fact_return', len(ret), ''),
    ('fact_review', len(rev), ''),
    ('bridge_item_promo', int(items.promo_id.notna().sum()+items.promo_id_2.notna().sum()), 'M:N'),
    ('fact_web_traffic', len(wt), 'chỉ 2013–2022'),
]
print(pd.DataFrame(sizes, columns=['bảng','số dòng','ghi chú']).to_string(index=False))

---

## 10 — Kết luận

**Toàn vẹn tham chiếu hoàn hảo** (0 orphan trên 14 quan hệ), nhưng có **10 vấn đề chất lượng dữ liệu**
cần thành rule khi load — xem `star_schema.md` §6.

**Bốn quyết định thiết kế bị dữ liệu ép buộc:**

1. `fact_order_item` **phải** có surrogate key — 16 cặp `(order_id, product_id)` trùng.
2. **Không có ship-to dimension** — `orders.zip` trùng `customers.zip` ở 100% dòng.
3. **Bỏ bảng `payments`** — `payment_value` suy ra được 100%, chỉ `installments` là mới.
4. **SCD Type 1 cho cả `dim_product` và `dim_customer`** — nguồn không có cột effective date,
   và `signup_date` thì không đáng tin (73,8% đơn phát sinh trước ngày đăng ký).

**Hai bẫy dễ mắc:**

- `review_title` trông như junk dimension, nhưng phụ thuộc hàm chạy theo chiều
  **`review_title → rating`** — mỗi title thuộc đúng một rating, còn một rating dùng 3–4 title.
  **Không** phải chiều ngược lại. Bảng tra cứu vì thế khóa theo title và có **18 dòng**,
  không phải `dim_rating` 5 dòng (`normalization.ipynb` §4.3).
- `products.price` có hai thang giá xen kẽ — là **lỗi dữ liệu**, đừng biến thành thuộc tính `price_tier`.
  Nhưng ranh giới là **100, không phải 1000** (§8.1): 654 SKU giá < 100 chưa từng bán một dòng nào,
  còn 34 SKU trong khoảng 100–1000 bán ở đúng giá niêm yết — **hàng rẻ hợp lệ**. Đặt ngưỡng ở 1000
  sẽ gắn cờ lỗi cho 30 sản phẩm bình thường, đồng thời gán con số "7,46% số dòng / 1,06% doanh thu"
  của chúng cho cả 688 SKU nghi lỗi.

Thiết kế đầy đủ, ERD và DDL nằm trong **`star_schema.md`**.